# PLUSVein RL850 — Dataset Statistics
Set the path in Cell 1, then run both cells. Paste the output back.

## Cell 1 · Set path

In [3]:
DATASET_DIR = r"F:\Palm Vein Multimodel V2\External\HandVein PLUSVein\HandVein_RL850"
IMG_EXTS = (".png",".jpg",".jpeg",".bmp",".tif",".tiff")

## Cell 2 · Count and report

In [4]:
import os, re
from collections import defaultdict

def parse_hand_identity(filename, folder):
    name = os.path.splitext(filename)[0]
    m = re.search(r"(\d{2,4})[_-]?([LR])\b", name) or \
        re.search(r"(\d{2,4}).*?([LR])[-_]?PALMAR", name, re.IGNORECASE)
    if m:
        return f"{m.group(1)}_{m.group(2).upper()}"
    side = "L" if re.search(r"\bL\b|left", name, re.IGNORECASE) else \
            ("R" if re.search(r"\bR\b|right", name, re.IGNORECASE) else "X")
    return f"{folder}_{side}"

def get_resolution(path):
    try:
        import cv2; img=cv2.imread(path)
        if img is not None:
            h,w=img.shape[:2]; return f"{w}x{h}"
    except Exception: pass
    try:
        from PIL import Image
        with Image.open(path) as im: return f"{im.size[0]}x{im.size[1]}"
    except Exception: return "unknown"

assert os.path.isdir(DATASET_DIR), f"Folder not found: {DATASET_DIR}"
by_hand=defaultdict(list); all_files=[]; subjects=set()
for dp,_,fns in os.walk(DATASET_DIR):
    folder=os.path.basename(dp)
    for fn in fns:
        if fn.lower().endswith(IMG_EXTS):
            full=os.path.join(dp,fn)
            hand=parse_hand_identity(fn,folder)
            by_hand[hand].append(full); all_files.append(full)
            subjects.add(hand.rsplit("_",1)[0])

assert all_files, f"No images found under {DATASET_DIR}"
counts=[len(v) for v in by_hand.values()]
res=get_resolution(all_files[0])

print("="*50)
print("PLUSVein RL850 — Dataset Statistics")
print("="*50)
print(f"Distinct subjects (persons) : {len(subjects)}")
print(f"Distinct hand-identities    : {len(by_hand)}")
print(f"Total images                : {len(all_files)}")
print(f"Images per hand (min/avg/max): {min(counts)} / {sum(counts)/len(counts):.1f} / {max(counts)}")
print(f"Sample resolution           : {res}")
print(f"Modality                    : NIR (RL850, 850 nm)")
print("="*50)
print("\nDistribution (images-per-hand : how many hands):")
dist=defaultdict(int)
for c in counts: dist[c]+=1
for k in sorted(dist): print(f"   {k} images : {dist[k]} hands")

PLUSVein RL850 — Dataset Statistics
Distinct subjects (persons) : 42
Distinct hand-identities    : 84
Total images                : 420
Images per hand (min/avg/max): 5 / 5.0 / 5
Sample resolution           : 1280x1024
Modality                    : NIR (RL850, 850 nm)

Distribution (images-per-hand : how many hands):
   5 images : 84 hands
